In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Smart MCQ Solver Challenge — DL & GenAI Project

**Roll No:** 23f2004827

## Objective
Predict the top-3 most likely correct answers (ranked) for multiple-choice
questions, evaluated using Mean Average Precision at 3 (MAP@3).

## Approach
Four models are built and compared:
1. TF-IDF + cosine similarity (baseline, no training)
2. Pretrained sentence-transformer embeddings + cosine similarity
3. Fine-tuned BERT (pretrained, fine-tuned for multiple-choice)
4. Feedforward neural network trained from scratch on TF-IDF features

Final predictions use a weighted ensemble of models 2–4, with weights
selected via grid search on a held-out validation set.

## 1. Data Loading
Loading train and test sets from the competition data.

In [7]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
train.head(2)

Train shape: (2000, 8)
Test shape: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A


## 2. Evaluation Metric
Implementing MAP@3 locally so we can evaluate models on our own
validation split before submitting to Kaggle. A quick sanity check
confirms the function behaves as expected on toy inputs.

In [8]:
def map_at_3(predictions, true_answers):
    
    scores = []
    for pred, true in zip(predictions, true_answers):
        pred_list = pred.split()
        if true in pred_list:
            rank = pred_list.index(true) + 1
            scores.append(1.0 / rank)
        else:
            scores.append(0.0)
    return np.mean(scores)


test_preds = ["A B C", "B A C", "C D A"]
test_true  = ["A", "A", "A"]
print("Sanity check (expect ~0.5):", map_at_3(test_preds, test_true))

Sanity check (expect ~0.5): 0.611111111111111


## 3. Model 1 — TF-IDF + Cosine Similarity (Baseline)
This baseline represents questions and answer options using TF-IDF vectors and ranks the options using cosine similarity.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_predict(df):
    
    option_cols = ['A', 'B', 'C', 'D', 'E']
    predictions = []

    
    all_text = df['prompt'].tolist()
    for col in option_cols:
        all_text += df[col].tolist()

    vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
    vectorizer.fit(all_text)

    prompt_vecs = vectorizer.transform(df['prompt'])

    for i in range(len(df)):
        option_texts = [df.iloc[i][col] for col in option_cols]
        option_vecs = vectorizer.transform(option_texts)
        sims = cosine_similarity(prompt_vecs[i], option_vecs)[0]  

        ranked_idx = np.argsort(sims)[::-1]  
        top3_letters = [option_cols[idx] for idx in ranked_idx[:3]]
        predictions.append(" ".join(top3_letters))

    return predictions


train_preds_model1 = tfidf_predict(train)
score_model1 = map_at_3(train_preds_model1, train['answer'].tolist())
print("Model 1 (TF-IDF) MAP@3 on train:", score_model1)

Model 1 (TF-IDF) MAP@3 on train: 0.27616666666666667


## 4. Experiment Tracking Setup
Logging in to Weights & Biases to track and compare MAP@3, accuracy,
and F1 across all models. The API key is loaded securely via Kaggle
Secrets rather than hardcoded.

In [10]:
!pip install wandb -q
import wandb
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
wandb_key = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f2004827 (23f2004827-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [11]:
run = wandb.init(project="23f2004827-t22026", name="model1-tfidf-baseline")
wandb.log({"map@3": score_model1, "model": "tfidf_cosine_similarity"})
run.finish()

map@3,▁
map@3,0.27617
model,tfidf_cosine_similar...


## 5. Model 2 — Pretrained Sentence-Transformer
Using `all-MiniLM-L6-v2`, a pretrained sentence embedding model, to
generate context-aware embeddings for the prompt and each option.
Options are ranked by cosine similarity in embedding space — a stronger
representation than TF-IDF since it captures semantic meaning, not just
word overlap.

In [12]:
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer


model_st = SentenceTransformer('all-MiniLM-L6-v2')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 106.4 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
import torch

def sentence_transformer_predict(df, model_st, batch_size=32):
   
    option_cols = ['A', 'B', 'C', 'D', 'E']
    predictions = []

    
    prompt_embeddings = model_st.encode(df['prompt'].tolist(), batch_size=batch_size, show_progress_bar=True, convert_to_tensor=True)

    for i in range(len(df)):
        option_texts = [df.iloc[i][col] for col in option_cols]
        option_embeddings = model_st.encode(option_texts, convert_to_tensor=True)

        
        sims = torch.nn.functional.cosine_similarity(
            prompt_embeddings[i].unsqueeze(0), option_embeddings
        )

        ranked_idx = torch.argsort(sims, descending=True).tolist()
        top3_letters = [option_cols[idx] for idx in ranked_idx[:3]]
        predictions.append(" ".join(top3_letters))

    return predictions

train_preds_model2 = sentence_transformer_predict(train, model_st)
score_model2 = map_at_3(train_preds_model2, train['answer'].tolist())
print("Model 2 (Sentence-Transformer) MAP@3 on train:", score_model2)

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Model 2 (Sentence-Transformer) MAP@3 on train: 0.4230833333333333


In [14]:
run = wandb.init(project="23f2004827-t22026", name="model2-sentence-transformer")
wandb.log({"map@3": score_model2, "model": "all-MiniLM-L6-v2_cosine_similarity"})
run.finish()

map@3,▁
map@3,0.42308
model,all-MiniLM-L6-v2_cos...


## 6. Model 3 — Fine-tuned BERT (Pretrained Model)
Formatting the MCQ task for `bert-base-uncased` using the multiple-choice
head, tokenizing (prompt, option) pairs for each of the 5 options, and
splitting into train/validation sets for fine-tuning.

In [15]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
import torch

MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
option_cols = ['A', 'B', 'C', 'D', 'E']

def preprocess(df):
    first_sentences = [[p]*5 for p in df['prompt']]
    second_sentences = [[df.iloc[i][col] for col in option_cols] for i in range(len(df))]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(first_sentences, second_sentences, truncation=True, padding='max_length', max_length=256)
    features = {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}
    return features

train_split, val_split = train_test_split(train, test_size=0.15, random_state=42)
train_split = train_split.reset_index(drop=True)
val_split = val_split.reset_index(drop=True)

train_features = preprocess(train_split)
val_features = preprocess(val_split)

print("Train samples:", len(train_split), "Val samples:", len(val_split))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train samples: 1700 Val samples: 300


In [16]:
class MCQDataset(torch.utils.data.Dataset):
    def __init__(self, features, labels=None):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features['input_ids'])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.features.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx])
        return item

label_map = {'A':0, 'B':1, 'C':2, 'D':3, 'E':4}
train_labels = [label_map[a] for a in train_split['answer']]
val_labels = [label_map[a] for a in val_split['answer']]

train_dataset = MCQDataset(train_features, train_labels)
val_dataset = MCQDataset(val_features, val_labels)

model3 = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

print("Dataset ready. Train size:", len(train_dataset), "Val size:", len(val_dataset))

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dataset ready. Train size: 1700 Val size: 300


## 7. Model 4 — Feedforward Neural Network (Built From Scratch)
This model is implemented from scratch using PyTorch. TF-IDF vectors are used as input features, and the network is trained using cross-entropy loss.

In [17]:
import torch
import torch.nn as nn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score

def top1_letter(pred_str):
    return pred_str.split()[0]

def evaluate_and_log(name, preds, true_answers, map3_score):
    top1_preds = [top1_letter(p) for p in preds]
    acc = accuracy_score(true_answers, top1_preds)
    f1 = f1_score(true_answers, top1_preds, average='macro')
    run = wandb.init(project="23f2004827-t22026", name=f"{name}-final-comparison")
    wandb.log({"map@3": map3_score, "accuracy": acc, "f1_macro": f1})
    run.finish()
    print(f"{name}: MAP@3={map3_score:.4f}, Accuracy={acc:.4f}, F1(macro)={f1:.4f}")


option_cols = ['A', 'B', 'C', 'D', 'E']
label_map_scratch = {'A':0, 'B':1, 'C':2, 'D':3, 'E':4}

vec_scratch = TfidfVectorizer(max_features=2000, stop_words='english')
all_text_scratch = train['prompt'].tolist()
for c in option_cols:
    all_text_scratch += train[c].tolist()
vec_scratch.fit(all_text_scratch)

def make_features(df):
    p = torch.tensor(vec_scratch.transform(df['prompt']).toarray(), dtype=torch.float32)
    opts = torch.stack([torch.tensor(vec_scratch.transform(df[c]).toarray(), dtype=torch.float32) for c in option_cols], dim=1)
    return p, opts

class ScratchMCQNet(nn.Module):
    def __init__(self, dim=2000, hidden=128):
        super().__init__()
        self.fc1 = nn.Linear(dim*2, hidden)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(hidden, 1)
    def forward(self, prompt_vec, option_vec):
        combined = torch.cat([prompt_vec, option_vec], dim=-1)
        x = self.relu(self.fc1(combined))
        x = self.dropout(x)
        return self.fc2(x).squeeze(-1)

p_train, o_train = make_features(train_split)
p_val, o_val = make_features(val_split)
labels_train = torch.tensor([label_map_scratch[a] for a in train_split['answer']])

model_scratch = ScratchMCQNet()
optimizer = torch.optim.Adam(model_scratch.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

p_train_exp = p_train.unsqueeze(1).expand(-1, 5, -1)
p_val_exp = p_val.unsqueeze(1).expand(-1, 5, -1)

for epoch in range(30):
    model_scratch.train()
    optimizer.zero_grad()
    logits = model_scratch(p_train_exp, o_train)
    loss = loss_fn(logits, labels_train)
    loss.backward()
    optimizer.step()
    if epoch % 5 == 0:
        print(f"Epoch {epoch}, train loss {loss.item():.4f}")

model_scratch.eval()
with torch.no_grad():
    val_logits = model_scratch(p_val_exp, o_val)
    top3_idx = torch.argsort(-val_logits, dim=1)[:, :3]
    preds_scratch = [" ".join([option_cols[i] for i in row.tolist()]) for row in top3_idx]

true_answers = val_split['answer'].tolist()
score_model1b = map_at_3(preds_scratch, true_answers)
print("Model 1 (from-scratch MLP) MAP@3 on validation:", score_model1b)

evaluate_and_log("model1-scratch-mlp", preds_scratch, true_answers, score_model1b)

Epoch 0, train loss 1.6097
Epoch 5, train loss 1.5836
Epoch 10, train loss 1.5499
Epoch 15, train loss 1.5083
Epoch 20, train loss 1.4569
Epoch 25, train loss 1.3990
Model 1 (from-scratch MLP) MAP@3 on validation: 0.9138888888888888


accuracy,▁
f1_macro,▁
map@3,▁
accuracy,0.84333
f1_macro,0.8492
map@3,0.91389


model1-scratch-mlp: MAP@3=0.9139, Accuracy=0.8433, F1(macro)=0.8492


In [18]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

CUDA available: True
Device: Tesla T4


## 8. Fine-tuning BERT
Training `bert-base-uncased` on the multiple-choice formatted dataset.

In [19]:
model3 = model3.to("cuda")

training_args = TrainingArguments(
    output_dir="./model3_output",
    eval_strategy="no",
    save_strategy="no",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=1,
    max_steps=150,
    logging_steps=25,
    report_to="none",
    fp16=True,
)

trainer = Trainer(
    model=model3,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
25,3.198295
50,3.011756
75,2.748467
100,2.532374
125,2.240587
150,2.049598


TrainOutput(global_step=150, training_loss=2.6301797739664714, metrics={'train_runtime': 496.4103, 'train_samples_per_second': 9.669, 'train_steps_per_second': 0.302, 'total_flos': 3120469098885120.0, 'train_loss': 2.6301797739664714, 'epoch': 2.7777777777777777})

In [20]:
import numpy as np

def predict_model3(trainer, features):
    dataset = MCQDataset(features)
    predictions = trainer.predict(dataset).predictions
    top3_idx = np.argsort(-predictions, axis=1)[:, :3]
    option_cols = ['A', 'B', 'C', 'D', 'E']
    return [" ".join([option_cols[i] for i in row]) for row in top3_idx]

val_preds_model3 = predict_model3(trainer, val_features)
score_model3 = map_at_3(val_preds_model3, val_split['answer'].tolist())
print("Model 3 (Fine-tuned BERT) MAP@3 on validation:", score_model3)

Model 3 (Fine-tuned BERT) MAP@3 on validation: 0.818888888888889


## 9. Model Comparison on Validation Set
Evaluating all models on the same held-out validation split using
MAP@3, accuracy, and macro F1, and logging results to W&B for
side-by-side comparison — as required by the experiment tracking
criteria.

In [21]:
val_true = val_split['answer'].tolist()

val_preds_m2 = sentence_transformer_predict(val_split, model_st)
score_model2_val = map_at_3(val_preds_m2, val_true)

evaluate_and_log("model2-sentence-transformer", val_preds_m2, val_true, score_model2_val)
evaluate_and_log("model3-finetuned-bert", val_preds_model3, val_true, score_model3)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

accuracy,▁
f1_macro,▁
map@3,▁
accuracy,0.26
f1_macro,0.2578
map@3,0.405


model2-sentence-transformer: MAP@3=0.4050, Accuracy=0.2600, F1(macro)=0.2578


accuracy,▁
f1_macro,▁
map@3,▁
accuracy,0.72667
f1_macro,0.71772
map@3,0.81889


model3-finetuned-bert: MAP@3=0.8189, Accuracy=0.7267, F1(macro)=0.7177


## 10. Generating Test Predictions & First Ensemble Attempt
Running all trained models on the test set and combining Models 1–3
into an initial weighted ensemble.

In [22]:




test_preds_m1 = tfidf_predict(test)


test_preds_m2 = sentence_transformer_predict(test, model_st)


test_features = preprocess(test)
import gc, torch
torch.cuda.empty_cache()
gc.collect()
model3.eval()
test_preds_m3 = predict_model3(trainer, test_features)


def ensemble_predict(preds_list, weights):
    option_cols = ['A', 'B', 'C', 'D', 'E']
    n = len(preds_list[0])
    final_preds = []
    for i in range(n):
        scores = {c: 0.0 for c in option_cols}
        for preds, w in zip(preds_list, weights):
            ranked = preds[i].split()
            for rank, letter in enumerate(ranked):
                scores[letter] += w * (1.0 / (rank + 1))  # higher weight to top rank
        top3 = sorted(scores, key=scores.get, reverse=True)[:3]
        final_preds.append(" ".join(top3))
    return final_preds


test_preds_ensemble = ensemble_predict(
    [test_preds_m1, test_preds_m2, test_preds_m3],
    weights=[0.1, 0.2, 0.7]
)


submission = pd.DataFrame({"ID": test["id"], "Prediction": test_preds_ensemble})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Submission saved. Rows:", len(submission))

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

   ID Prediction
0   1      A B D
1   2      B C A
2   3      B C E
3   4      E A C
4   5      D C E
Submission saved. Rows: 500


## 11. Weighted Ensemble with Grid Search
Adding Model 4 (from-scratch MLP) to the ensemble and grid-searching
combination weights on the validation set to find the best-performing
blend of scratch MLP, sentence-transformer, and BERT predictions.

In [23]:
import torch
import numpy as np


p_test, o_test = make_features(test)
p_test_exp = p_test.unsqueeze(1).expand(-1, 5, -1)

model_scratch.eval()
with torch.no_grad():
    test_logits = model_scratch(p_test_exp, o_test)
    top3_idx_test = torch.argsort(-test_logits, dim=1)[:, :3]
    test_preds_scratch = [" ".join([option_cols[i] for i in row.tolist()]) for row in top3_idx_test]
print("Scratch MLP test predictions ready:", len(test_preds_scratch))


test_preds_m2_new = sentence_transformer_predict(test, model_st)
test_features_new = preprocess(test)
test_preds_m3_new = predict_model3(trainer, test_features_new)
print("Model 2 and Model 3 test predictions ready.")


val_true = val_split['answer'].tolist()
weight_options = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

best_score, best_weights = -1, None
for w1 in weight_options:        # scratch MLP
    for w3 in weight_options:    # BERT
        w2 = round(1.0 - w1 - w3, 2)   # sentence-transformer
        if w2 < 0 or w2 > 1:
            continue
        preds = ensemble_predict([preds_scratch, val_preds_m2, val_preds_model3], weights=[w1, w2, w3])
        score = map_at_3(preds, val_true)
        if score > best_score:
            best_score, best_weights = score, (w1, w2, w3)

print("Best weights (scratch, sentence-transformer, bert):", best_weights)
print("Best validation MAP@3:", best_score)


final_test_preds = ensemble_predict(
    [test_preds_scratch, test_preds_m2_new, test_preds_m3_new],
    weights=list(best_weights)
)

submission = pd.DataFrame({"ID": test["id"], "Prediction": final_test_preds})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("New submission saved. Rows:", len(submission))

Scratch MLP test predictions ready: 500


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Model 2 and Model 3 test predictions ready.
Best weights (scratch, sentence-transformer, bert): (0.6, 0.2, 0.2)
Best validation MAP@3: 0.9188888888888888
   ID Prediction
0   1      E A C
1   2      B C A
2   3      B E D
3   4      E A C
4   5      C D A
New submission saved. Rows: 500


## 12. Experiment: BERT-Only Submission
Testing a single-model (BERT-only) submission as a comparison point
against the ensembled approach.

In [24]:

test_features_pure = preprocess(test)
test_preds_bert_only = predict_model3(trainer, test_features_pure)

submission = pd.DataFrame({"ID": test["id"], "Prediction": test_preds_bert_only})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Pure BERT submission saved. Rows:", len(submission))

   ID Prediction
0   1      A D B
1   2      B C A
2   3      B E C
3   4      E A C
4   5      D E C
Pure BERT submission saved. Rows: 500


## 13. Experiment: Probability-Weighted Ensemble
Trying an alternative ensembling strategy using softmax probabilities
instead of rank-based scoring, to see if it generalizes better.

In [25]:
import numpy as np

option_cols = ['A', 'B', 'C', 'D', 'E']

def softmax_np(x):
    e = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e / np.sum(e, axis=1, keepdims=True)


def tfidf_scores(df):
    all_text = df['prompt'].tolist()
    for col in option_cols:
        all_text += df[col].tolist()
    vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
    vectorizer.fit(all_text)
    prompt_vecs = vectorizer.transform(df['prompt'])
    scores = np.zeros((len(df), 5))
    for i in range(len(df)):
        option_texts = [df.iloc[i][col] for col in option_cols]
        option_vecs = vectorizer.transform(option_texts)
        scores[i] = cosine_similarity(prompt_vecs[i], option_vecs)[0]
    return scores

def st_scores(df, model_st, batch_size=32):
    prompt_embeddings = model_st.encode(df['prompt'].tolist(), batch_size=batch_size, convert_to_tensor=True)
    scores = np.zeros((len(df), 5))
    for i in range(len(df)):
        option_texts = [df.iloc[i][col] for col in option_cols]
        option_embeddings = model_st.encode(option_texts, convert_to_tensor=True)
        sims = torch.nn.functional.cosine_similarity(prompt_embeddings[i].unsqueeze(0), option_embeddings)
        scores[i] = sims.cpu().numpy()
    return scores

def bert_scores(trainer, features):
    dataset = MCQDataset(features)
    return trainer.predict(dataset).predictions

def scores_to_top3(combined_scores):
    top3_idx = np.argsort(-combined_scores, axis=1)[:, :3]
    return [" ".join([option_cols[i] for i in row]) for row in top3_idx]


val_true = val_split['answer'].tolist()
val_tfidf_probs = softmax_np(tfidf_scores(val_split))
val_st_probs = softmax_np(st_scores(val_split, model_st))
val_bert_probs = softmax_np(bert_scores(trainer, val_features))

best_score, best_weights = -1, None
weight_options = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
for w1 in weight_options:
    for w3 in weight_options:
        w2 = round(1.0 - w1 - w3, 2)
        if w2 < 0 or w2 > 1:
            continue
        combined = w1*val_tfidf_probs + w2*val_st_probs + w3*val_bert_probs
        score = map_at_3(scores_to_top3(combined), val_true)
        if score > best_score:
            best_score, best_weights = score, (w1, w2, w3)

print("Best weights (tfidf, sentence-transformer, bert):", best_weights)
print("Best validation MAP@3 (probability-based ensemble):", best_score)


test_features_new = preprocess(test)
test_tfidf_probs = softmax_np(tfidf_scores(test))
test_st_probs = softmax_np(st_scores(test, model_st))
test_bert_probs = softmax_np(bert_scores(trainer, test_features_new))

w1, w2, w3 = best_weights
combined_test = w1*test_tfidf_probs + w2*test_st_probs + w3*test_bert_probs
final_preds = scores_to_top3(combined_test)

submission = pd.DataFrame({"ID": test["id"], "Prediction": final_preds})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("Probability-based ensemble submission saved. Rows:", len(submission))

Best weights (tfidf, sentence-transformer, bert): (0.0, 0.3, 0.7)
Best validation MAP@3 (probability-based ensemble): 0.8366666666666668


   ID Prediction
0   1      A D B
1   2      B C A
2   3      B A C
3   4      E A C
4   5      D E C
Probability-based ensemble submission saved. Rows: 500


## 14. Experiment: DeBERTa-v3 as a Second Transformer Model
Testing whether a different pretrained transformer architecture
(DeBERTa-v3-small) improves results. This was trained with limited
steps under time constraints and underperformed BERT (MAP@3 0.359 vs
0.819), likely due to insufficient training — included here as a
documented negative result rather than discarded, since comparing
model families was part of the goal.

In [26]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer


MODEL_NAME_V2 = "microsoft/deberta-v3-small"
tokenizer_v2 = AutoTokenizer.from_pretrained(MODEL_NAME_V2)

def preprocess_v2(df):
    first_sentences = [[p]*5 for p in df['prompt']]
    second_sentences = [[df.iloc[i][col] for col in option_cols] for i in range(len(df))]
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    tokenized = tokenizer_v2(first_sentences, second_sentences, truncation=True, padding='max_length', max_length=256)
    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}

train_features_v2 = preprocess_v2(train_split)
val_features_v2 = preprocess_v2(val_split)
train_dataset_v2 = MCQDataset(train_features_v2, train_labels)
val_dataset_v2 = MCQDataset(val_features_v2, val_labels)

model3_v2 = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME_V2, hidden_dropout_prob=0.3, attention_probs_dropout_prob=0.3)
model3_v2 = model3_v2.to("cuda")

training_args_v2 = TrainingArguments(
    output_dir="./model3_v2_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.1,
    warmup_ratio=0.1,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    fp16=False,
)

trainer_v2 = Trainer(model=model3_v2, args=training_args_v2, train_dataset=train_dataset_v2, eval_dataset=val_dataset_v2)
trainer_v2.train()

val_preds_model3_v2 = predict_model3(trainer_v2, val_features_v2)
score_model3_v2 = map_at_3(val_preds_model3_v2, val_split['answer'].tolist())
print("Model 3 v2 (DeBERTa, regularized, early-stopped) MAP@3 on validation:", score_model3_v2)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.layer.3.output.LayerNorm.weight', 'deberta.encoder.layer.3.output.Laye

Model 3 v2 (DeBERTa, regularized, early-stopped) MAP@3 on validation: 0.3588888888888889


## 15. Final Submission

The final submission is generated using the best-performing model or ensemble based on validation performance. The predictions are saved as `submission.csv` for Kaggle submission.

In [27]:

final_test_preds = ensemble_predict(
    [test_preds_scratch, test_preds_m2, test_preds_m3],
    weights=[0.6, 0.2, 0.2]
)

submission = pd.DataFrame({"ID": test["id"], "Prediction": final_test_preds})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("FINAL best submission saved. Rows:", len(submission))

   ID Prediction
0   1      E A C
1   2      B C A
2   3      B E D
3   4      E A C
4   5      C D A
FINAL best submission saved. Rows: 500


## 16. Model 5 — Zero-shot LLM (Qwen2.5-7B) using answer-letter logits
The options here are mostly reworded versions of the same fact, so a small
model trained on just 2000 rows can't really tell which one is true — it
has no actual knowledge of physics/philosophy facts, just pattern
matching. So instead of training anything, this model uses a bigger
pretrained instruction-tuned LLM and just reads off its confidence for
each letter (A-E) directly from the logits, no fine-tuning involved.

In [1]:
!pip install -q -U bitsandbytes accelerate transformers
 
import os, gc, re
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 47.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 102.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 86.3 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have num

### 16.1 Reusing train/test and option labels
Same train/test data as before, just defining OPTS again for this section.

In [2]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
print(train.shape, test.shape)

OPTS = ["A", "B", "C", "D", "E"]

(2000, 8) (500, 7)


### 16.2 Loading Qwen2.5-7B-Instruct in 4-bit
Using bitsandbytes to load it in 4-bit so it fits on the Kaggle GPU.
Also need to find out what token id each letter (A-E) maps to in the
tokenizer, since that's what we'll read the logits for later.

In [9]:
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
 
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
 

def letter_token_id(letter):
    candidates = [letter, " " + letter]
    for c in candidates:
        ids = tokenizer.encode(c, add_special_tokens=False)
        if len(ids) == 1:
            return ids[0]
   
    return tokenizer.encode(candidates[0], add_special_tokens=False)[-1]
 
letter_ids = {letter: letter_token_id(letter) for letter in OPTS}
print("Letter token ids:", letter_ids)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Letter token ids: {'A': 32, 'B': 33, 'C': 34, 'D': 35, 'E': 36}


### 16.3 Building the prompt
Formats the question + all 5 options as a chat prompt, and ends it with
"The correct answer is: " so the model's very next predicted token is
the answer letter — that's what lets us score all 5 options directly.

In [4]:
def build_prompt(row):
    q = row["prompt"]
    opts_txt = "\n".join(f"{L}. {row[L]}" for L in OPTS)
    messages = [
        {"role": "system", "content": "You are an expert across physics, philosophy, "
                                       "astronomy and general science. Answer multiple "
                                       "choice questions by selecting the single best "
                                       "option letter."},
        {"role": "user", "content": f"Question: {q}\n\n{opts_txt}\n\n"
                                     f"Reply with ONLY the letter of the correct answer."},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    text += "The correct answer is: "
    return text

### 16.4 Scoring function
One forward pass per batch, no text generation needed. Pull out the
logits at the last token, keep only the 5 letter positions, softmax
them, and rank to get the top-3 for MAP@3.

In [10]:
@torch.no_grad()
def score_batch(rows, batch_size=8):
    all_top3 = []
    for i in range(0, len(rows), batch_size):
        chunk = rows.iloc[i:i + batch_size]
        prompts = [build_prompt(r) for _, r in chunk.iterrows()]
        enc = tokenizer(prompts, return_tensors="pt", padding=True).to(DEVICE)
        out = model(**enc)

        last_logits = out.logits[:, -1, :]  # (B, vocab)

        for row_logits in last_logits:
            letter_logits = torch.tensor(
                [row_logits[letter_ids[L]].item() for L in OPTS]
            )
            probs = torch.softmax(letter_logits, dim=0)
            ranked = [OPTS[j] for j in torch.argsort(probs, descending=True)]
            all_top3.append(" ".join(ranked[:3]))
        del enc, out
        gc.collect(); torch.cuda.empty_cache()
    return all_top3

### 16.5 Checking MAP@3 on a validation sample
Testing on 300 rows from train before running the full test set, same
as done for the other models.

In [11]:
def map_at_3(preds, truths):
    scores = []
    for pred, true in zip(preds, truths):
        pl = pred.split()
        scores.append(1.0 / (pl.index(true) + 1) if true in pl else 0.0)
    return np.mean(scores)
 
val = train.sample(n=300, random_state=42).reset_index(drop=True)
val_preds = score_batch(val, batch_size=8)
print("Validation MAP@3:", map_at_3(val_preds, val["answer"].tolist()))
 


Validation MAP@3: 0.8916666666666667


### 16.6 Logging to W&B
Adding this run to the same W&B project as the other models so it shows
up in the comparison.

In [15]:
import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")  
wandb.login(key=wandb_api_key)

wandb.init(project="23f2004827-t22026", name="qwen2.5-7b-instruct-zeroshot-logit")
wandb.log({"val_map_at_3": map_at_3(val_preds, val["answer"].tolist())})
wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


val_map_at_3,▁
val_map_at_3,0.89167


## 17. Final test predictions — Model 5
Running this on the full test set and saving submission.csv.

In [16]:
test_preds = score_batch(test, batch_size=8)
submission = pd.DataFrame({"ID": test["id"], "Prediction": test_preds})
submission.to_csv("submission.csv", index=False)
submission.head()

,ID,Prediction
0,1,A E D
1,2,B A E
2,3,C B E
3,4,E A B
4,5,C A D
